# RetailLens: customer behavior to repeat-purchase prediction
An independent case study using **UCI Online Retail**. The objective is to understand customers, rank likely repeat purchasers, and propose measurable business experiments.

Run `python -m retaillens.pipeline` before this notebook. Use a Python kernel with the project dependencies installed.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
import numpy as np
import plotly.express as px
from IPython.display import display

ROOT = Path.cwd() if (Path.cwd() / 'retaillens').is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from retaillens.pipeline import clean_transactions, customer_features, labelled_snapshot, temporal_split, FEATURES
DATA = ROOT / 'artifacts' / 'dashboard'
assert (DATA / 'summary.json').exists(), 'Run python -m retaillens.pipeline first.'
summary = json.loads((DATA / 'summary.json').read_text(encoding='utf-8'))
evaluation = json.loads((DATA / 'evaluation.json').read_text(encoding='utf-8'))
quality = json.loads((DATA / 'quality.json').read_text(encoding='utf-8'))
sales = pd.read_csv(DATA / 'transactions.csv', dtype={'CustomerID': 'string', 'InvoiceNo': str, 'StockCode': str}, parse_dates=['InvoiceDate'])
customers = pd.read_csv(DATA / 'customers.csv', dtype={'CustomerID': str})
print(f"{len(sales):,} cleaned positive-sales lines; {sales.CustomerID.nunique():,} identifiable customers")

524,878 cleaned positive-sales lines; 4,338 identifiable customers


## 1. Audit before analysis
Exact duplicate rows, cancellation invoices, invalid timestamps and non-positive quantity/price are excluded. This defines **gross positive sales, not net revenue**. Anonymous purchases remain in sales analysis but not customer modeling. Exact-row deduplication is a heuristic.

In [2]:
display(pd.DataFrame(quality['steps']))
assert quality['raw_rows'] - sum(step['removed'] for step in quality['steps']) == quality['clean_sales_rows']
print(f"Sales lines without customer IDs: {quality['rows_without_customer_id']:,}")
print(f"Gross positive sales: £{sales.Revenue.sum():,.2f}")
display(sales[['Quantity', 'Revenue']].describe())

Sales lines without customer IDs: 132,186
Gross positive sales: £10,642,110.80


,reason,removed,remaining
0,Exact duplicate rows (documented heuristic),5268,536641
1,Invalid transaction timestamp,0,536641
2,Missing invoice or product identifier,0,536641
3,Cancelled invoices,9251,527390
4,Invalid / non-positive quantity or unit price,2512,524878


,Quantity,Revenue
count,524878.000000,524878.000000
mean,10.616600,20.275399
std,156.280031,271.693566
min,1.000000,0.001000
25%,1.000000,3.900000
50%,4.000000,9.920000
75%,11.000000,17.700000
max,80995.000000,168469.600000


## 2. Sales patterns and observed-cohort retention
December 2011 is partial, so its total cannot be compared directly with complete months. Cohorts start at the first **observed** purchase; blank future months are unobservable, not zero retention.

In [3]:
monthly = pd.read_csv(DATA / 'monthly.csv')
display(monthly)
display(px.line(monthly, x='month', y='revenue', markers=True, title='Monthly positive sales (£); final month is partial', template='plotly_dark'))
cohorts = pd.read_csv(DATA / 'cohorts.csv').set_index('cohort')
display(px.imshow(cohorts, aspect='auto', zmin=0, zmax=1, title='Observed-cohort retention', color_continuous_scale='Greens'))

,month,revenue,orders,customers
0,2010-12,821452.730,1559,885
1,2011-01,689811.610,1086,741
2,2011-02,522545.560,1100,758
3,2011-03,716215.260,1454,974
4,2011-04,536968.491,1246,856
5,2011-05,769296.610,1681,1056
6,2011-06,760547.010,1533,991
7,2011-07,718076.121,1475,949
8,2011-08,757841.380,1361,935
9,2011-09,1056435.192,1837,1266


Figure({
    'data': [{'hovertemplate': 'month=%{x}<br>revenue=%{y}<extra></extra>',
              'legendgroup': '',
              'line': {'color': '#636efa', 'dash': 'solid'},
              'marker': {'symbol': 'circle'},
              'mode': 'lines+markers',
              'name': '',
              'orientation': 'v',
              'showlegend': False,
              'type': 'scatter',
              'x': array(['2010-12', '2011-01', '2011-02', '2011-03', '2011-04', '2011-05',
                          '2011-06', '2011-07', '2011-08', '2011-09', '2011-10', '2011-11',
                          '2011-12'], dtype=object),
              'xaxis': 'x',
              'y': {'bdata': ('XI/CdZkRKUGF61E4Jw0lQdejcD3G5B' ... 'G6H5ExQXsUrsd68jZBj8L1qLx2I0E='),
                    'dtype': 'f8'},
              'yaxis': 'y'}],
    'layout': {'legend': {'tracegroupgap': 0},
               'template': '...',
               'title': {'text': 'Monthly positive sales (£); final month is partial'},
      

Figure({
    'data': [{'coloraxis': 'coloraxis',
              'hovertemplate': 'x: %{x}<br>cohort: %{y}<br>color: %{z}<extra></extra>',
              'name': '0',
              'type': 'heatmap',
              'x': array(['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12'],
                         dtype=object),
              'xaxis': 'x',
              'y': array(['2010-12', '2011-01', '2011-02', '2011-03', '2011-04', '2011-05',
                          '2011-06', '2011-07', '2011-08', '2011-09', '2011-10', '2011-11',
                          '2011-12'], dtype=object),
              'yaxis': 'y',
              'z': {'bdata': ('AAAAAAAA8D+39KrNNW7XP94j9GS3rt' ... 'AAAAD4fwAAAAAAAPh/AAAAAAAA+H8='),
                    'dtype': 'f8',
                    'shape': '13, 13'}}],
    'layout': {'coloraxis': {'cmax': 1,
                             'cmin': 0,
                             'colorscale': [[0.0, 'rgb(247,252,245)'], [0.125,
                                     

## 3. Customer segmentation
Recency is time since last purchase; frequency counts unique invoices; monetary is positive spend. All use the same final **180-day** window. K-means operates on standardized `log1p(RFM)`. Candidate k=2…6 values are compared by sampled silhouette. Descriptive names do not prove business value.

In [4]:
cutoff = pd.Timestamp(summary['snapshot'])
rfm = customer_features(sales, cutoff)
assert rfm.CustomerID.nunique() == len(customers)
segmentation = json.loads((DATA / 'segmentation.json').read_text(encoding='utf-8'))
display(pd.DataFrame(segmentation['candidates']))
display(customers.groupby('segment').agg(customers=('CustomerID', 'count'), median_recency=('recency', 'median'), median_orders=('frequency', 'median'), median_spend=('monetary', 'median')).round(2))
display(px.scatter(customers, x='recency', y='monetary', color='segment', log_y=True, opacity=.5, title='Final-window customer behavior', template='plotly_dark'))

,k,silhouette
0,2,0.430423
1,3,0.334762
2,4,0.331003
3,5,0.293566
4,6,0.283436


,customers,median_recency,median_orders,median_spend
segment,,,,
Active loyalists · 02,1222,10.46,4.0,1559.82
Occasional shoppers · 01,2255,57.47,1.0,347.75


Figure({
    'data': [{'hovertemplate': 'segment=Active loyalists · 02<br>recency=%{x}<br>monetary=%{y}<extra></extra>',
              'legendgroup': 'Active loyalists · 02',
              'marker': {'color': '#636efa', 'opacity': 0.5, 'symbol': 'circle'},
              'mode': 'markers',
              'name': 'Active loyalists · 02',
              'showlegend': True,
              'type': 'scattergl',
              'x': {'bdata': ('YQu2YAu2AkBhC7ZgC6pAQLZgC7ZgC/' ... 'mf9Hk4QC7Ygi3YwgNA0id90if9C0A='),
                    'dtype': 'f8'},
              'xaxis': 'x',
              'y': {'bdata': ('mpmZmRlwoEBSuB6Fqz+4QArXo3A9QJ' ... 'MzM6meQK5H4XoUZKBA16NwPQrol0A='),
                    'dtype': 'f8'},
              'yaxis': 'y'},
             {'hovertemplate': 'segment=Occasional shoppers · 01<br>recency=%{x}<br>monetary=%{y}<extra></extra>',
              'legendgroup': 'Occasional shoppers · 01',
              'marker': {'color': '#EF553B', 'opacity': 0.5, 'symbol': 'circle'},
       

## 4. Labels and chronology
Features use `[cutoff − 180 days, cutoff)`; labels use `[cutoff, cutoff + 30 days)`. Customers active in the prior window are eligible. An observed positive purchase is the target; permanent churn is not observed. Training label windows must end before validation, and validation labels before test. Repeated customers across snapshots are intentional for an existing-customer task.

In [5]:
example_cutoff = pd.Timestamp(evaluation['split']['test']['first_snapshot'])
example = labelled_snapshot(sales, example_cutoff, sales.InvoiceDate.max())
display(example[['snapshot', 'label_end', 'repeat_purchase']].head())
print(f'Test snapshot purchase rate: {example.repeat_purchase.mean():.1%}')
display(pd.DataFrame(evaluation['split']).T)
assert pd.Timestamp(evaluation['split']['train']['last_label_end']) <= pd.Timestamp(evaluation['split']['validation']['first_snapshot'])
assert pd.Timestamp(evaluation['split']['validation']['last_label_end']) <= pd.Timestamp(evaluation['split']['test']['first_snapshot'])
assert 'CustomerID' not in FEATURES

Test snapshot purchase rate: 38.4%


,snapshot,label_end,repeat_purchase
0,2011-11-01,2011-12-01,0
1,2011-11-01,2011-12-01,0
2,2011-11-01,2011-12-01,1
3,2011-11-01,2011-12-01,0
4,2011-11-01,2011-12-01,0


,rows,customers,first_snapshot,last_snapshot,last_label_end
train,10926,3267,2011-06-01,2011-09-01,2011-10-01
validation,2925,2925,2011-10-01,2011-10-01,2011-10-31
test,3194,3194,2011-11-01,2011-11-01,2011-12-01


## 5. Compare with a baseline
A prior-probability baseline, logistic regression and histogram gradient boosting compete on **validation average precision**. Preprocessing is trained within the supervised pipeline. Candidates are refitted on train + validation only after selection, then compared on the held-out test. ROC AUC, Brier score and top-10% lift complement average precision. Permutation importance is diagnostic, not causal.

In [6]:
rows = []
for name, scores in evaluation['models'].items():
    rows.append({'model': name, 'validation_AP': scores['validation']['average_precision'], **scores['test']})
results = pd.DataFrame(rows).set_index('model')
display(results.round(4))
print('Selected on validation:', evaluation['selected_model'])
display(px.bar(results.reset_index(), x='model', y='average_precision', title='Held-out average precision', template='plotly_dark'))
display(pd.DataFrame(evaluation['permutation_importance']).sort_values('mean', ascending=False).round(4))

Selected on validation: Gradient boosting


,validation_AP,average_precision,roc_auc,brier_score,precision_at_10pct,lift_at_10pct,positive_rate,rows
model,,,,,,,,
Prior baseline,0.3087,0.3838,0.5000,0.2457,0.4031,1.0502,0.3838,3194
Logistic regression,0.5554,0.6539,0.7180,0.2080,0.8344,2.1737,0.3838,3194
Gradient boosting,0.5596,0.6592,0.7201,0.2056,0.8438,2.1982,0.3838,3194


Figure({
    'data': [{'hovertemplate': 'model=%{x}<br>average_precision=%{y}<extra></extra>',
              'legendgroup': '',
              'marker': {'color': '#636efa', 'pattern': {'shape': ''}},
              'name': '',
              'orientation': 'v',
              'showlegend': False,
              'textposition': 'auto',
              'type': 'bar',
              'x': array(['Prior baseline', 'Logistic regression', 'Gradient boosting'],
                         dtype=object),
              'xaxis': 'x',
              'y': {'bdata': 'qXXKZemQ2D8Ycsu06ezkP9r20BiQGOU/', 'dtype': 'f8'},
              'yaxis': 'y'}],
    'layout': {'barmode': 'relative',
               'legend': {'tracegroupgap': 0},
               'template': '...',
               'title': {'text': 'Held-out average precision'},
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'model'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': 'average_precisi

,feature,mean,std
1,frequency,0.2166,0.0115
0,recency,0.0167,0.0028
4,tenure_days,0.0061,0.0011
6,units,0.0056,0.0020
5,unique_products,0.0046,0.0010
2,monetary,0.0024,0.0012
3,average_order_value,0.0001,0.0006


## 6. Final scores and business experiments
Final December scores have no complete future labels. They are estimates from the previously selected model, not measured December performance. Probability estimates have not been separately calibrated. Likely purchasers are not necessarily the people most influenced by an offer.

Suggested next step: run controlled loyalty and reactivation experiments, tracking **incremental purchases and margin** with randomized holdouts. Historical association alone cannot establish campaign impact.

In [7]:
assert customers.repeat_probability.between(0, 1).all()
display(customers.sort_values('repeat_probability', ascending=False)[['CustomerID', 'segment', 'recency', 'frequency', 'monetary', 'repeat_probability']].head(10))
print((ROOT / 'reports' / 'findings.md').read_text(encoding='utf-8'))

# RetailLens — findings from the executed pipeline

This historical case study uses UCI Online Retail, not live business data.

## Data and revenue

- Raw transaction lines: **541,909**; cleaned positive-sales lines: **524,878**.
- Gross positive sales: **£10,642,110.80**, across **19,960** invoices.
- Customers with identifiable IDs: **4,338**. **132,186** sales lines lack a customer ID; they contribute to sales totals but not customer modeling.
- Returns, cancellations and non-positive prices/quantities are excluded. Sales are not net revenue or profit.

## Customer segmentation

- Segmentation uses the final 180-day window, ending at 2011-12-10.
- Selected **2 clusters**, silhouette **0.430**, from k=2…6 candidates.
- Highest aggregate spend segment: **Active loyalists · 02**, with **1,222** customers and **£4,222,028.44** in that window.
- Segment labels describe observed behavior. They are analyst interpretations, not ground-truth personas.

## Held-out prediction results

- Selec

,CustomerID,segment,recency,frequency,monetary,repeat_probability
1511,14911,Active loyalists · 02,1.337500,135,100326.53,0.948111
1356,14646,Active loyalists · 02,1.491667,46,164070.10,0.948111
3228,17841,Active loyalists · 02,1.495139,74,27723.49,0.948111
1745,15311,Active loyalists · 02,0.500000,46,27022.42,0.948111
450,13089,Active loyalists · 02,2.623611,43,29294.91,0.948111
269,12748,Active loyalists · 02,0.486111,117,23183.25,0.948111
1335,14606,Active loyalists · 02,1.188889,42,5013.67,0.945611
1286,14527,Active loyalists · 02,2.486806,28,4944.77,0.944951
553,13263,Active loyalists · 02,1.417361,20,3996.53,0.943641
1075,14156,Active loyalists · 02,9.545833,36,72359.73,0.938070


## Attribution and limitations
Chen, D. (2015). *Online Retail*. UCI Machine Learning Repository. [DOI: 10.24432/C5BW33](https://doi.org/10.24432/C5BW33). **CC BY 4.0**.

One UK retailer in 2010–2011, including wholesale customers. Exact duplicates may include legitimate repeated lines; some product codes are charges. Customer tenure is measured only inside the 180-day lookback. Country filters do not refit global models. See the README and findings report for complete assumptions.